In [ ]:
#!/usr/bin/env python3
"""
v5 G0 — VCC leg. STEP 0: SCHEMA PROBE (§10 of docs/TASK_v5_G0_VCC_leg.md).

RUN THIS FIRST. It reads only .obs, .var and a small row slice of .X.
Target runtime: < 10 minutes. It must NOT stream the whole 15 GB.

Environment (Kaggle, CPU only, free tier):
    pre-installed: numpy, pandas, h5py, scipy
    no pip installs required. anndata is deliberately NOT used (it would try to
    materialise the whole matrix on several code paths).

Input : /kaggle/input/<DATASET_SLUG>/adata_Training.h5ad
Output: /kaggle/working/v5g0_vcc_probe_v1.json
        /kaggle/working/v5g0_vcc_percell_meta_v1.csv.gz   (cell -> target_gene, batch)

WHAT IT ANSWERS (these drive the whole preprocessing path of STEP 1):
  1. is .X raw counts or already log-normalised?
  2. dense or sparse; if sparse, CSR or CSC; dtype; sparsity
  3. .obs.target_gene value counts -> 150 + non-targeting? true cells-per-pert
     distribution (NOT the 1,221 average)
  4. does .obs.batch confound perturbation? (contingency, per-pert batch entropy)
  5. .var namespaces present
"""
import gzip
import json
import os
import sys

import h5py
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- config ----
H5AD = os.environ.get("VCC_H5AD", "")
if not H5AD:
    # auto-discover under /kaggle/input
    for root, _dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".h5ad"):
                H5AD = os.path.join(root, f)
                break
        if H5AD:
            break
if not H5AD or not os.path.exists(H5AD):
    sys.exit("FATAL: no .h5ad found. Set env VCC_H5AD=/kaggle/input/<slug>/adata_Training.h5ad")

OUT = os.environ.get("VCC_OUT", "/kaggle/working")
os.makedirs(OUT, exist_ok=True)
PROBE_ROWS = 2000          # row slice used for the raw-vs-lognorm call

report = {"file": H5AD, "file_gb": round(os.path.getsize(H5AD) / 1e9, 2)}


# ------------------------------------------------------- h5ad primitives ----
def read_str_array(h5, key):
    """Read an .obs column that may be a plain string array or a categorical."""
    node = h5[key]
    if isinstance(node, h5py.Group):                     # categorical
        cats = node["categories"][:]
        codes = node["codes"][:]
        cats = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in cats])
        out = np.where(codes >= 0, cats[np.clip(codes, 0, None)], "<NA>")
        return out
    v = node[:]
    if v.dtype.kind in "SO":
        return np.array([x.decode() if isinstance(x, bytes) else str(x) for x in v])
    return v.astype(str)


def obs_columns(h5):
    g = h5["obs"]
    cols = list(g.attrs.get("column-order", []))
    cols = [c.decode() if isinstance(c, bytes) else str(c) for c in cols]
    if not cols:
        cols = [k for k in g.keys() if k != "_index"]
    return cols


with h5py.File(H5AD, "r") as h5:
    # ---- 1. matrix layout ---------------------------------------------------
    X = h5["X"]
    if isinstance(X, h5py.Group):
        enc = X.attrs.get("encoding-type", b"")
        enc = enc.decode() if isinstance(enc, bytes) else str(enc)
        layout = "csr" if "csr" in enc else ("csc" if "csc" in enc else enc or "sparse?")
        shape = tuple(int(v) for v in X.attrs["shape"])
        nnz = int(X["data"].shape[0])
        dtype = str(X["data"].dtype)
        idx_dtype = str(X["indices"].dtype)
        density = nnz / (shape[0] * shape[1])
        sample = X["data"][: min(2_000_000, nnz)]
    else:
        layout = "dense"
        shape = tuple(int(v) for v in X.shape)
        nnz = None
        dtype = str(X.dtype)
        idx_dtype = None
        blk = X[:PROBE_ROWS, :]
        density = float((blk != 0).mean())
        sample = blk[blk != 0].ravel()
        del blk

    report["matrix"] = {
        "layout": layout, "shape": list(shape), "dtype": dtype,
        "indices_dtype": idx_dtype, "nnz": nnz,
        "density_estimate": round(float(density), 4),
    }

    # ---- 2. raw counts or log-normalised? ----------------------------------
    s = np.asarray(sample, dtype=np.float64)
    s = s[np.isfinite(s)]
    frac_integer = float(np.mean(np.abs(s - np.round(s)) < 1e-6)) if s.size else np.nan
    report["value_distribution"] = {
        "n_sampled_nonzero": int(s.size),
        "min": float(s.min()) if s.size else None,
        "max": float(s.max()) if s.size else None,
        "mean": float(s.mean()) if s.size else None,
        "quantiles": {q: float(np.quantile(s, q)) for q in (0.5, 0.9, 0.99, 0.999)} if s.size else {},
        "fraction_integer_valued": round(frac_integer, 6),
        "n_distinct_in_sample": int(np.unique(s[: 200_000]).size) if s.size else 0,
    }
    if frac_integer > 0.999:
        xcall = "RAW_COUNTS"
    elif s.size and s.max() < 20 and frac_integer < 0.5:
        xcall = "LOG_NORMALIZED"
    else:
        xcall = "AMBIGUOUS_INSPECT_MANUALLY"
    report["X_call"] = xcall

    # ---- 3. per-cell library size on the probe slice ------------------------
    if layout == "dense":
        blk = np.asarray(h5["X"][:PROBE_ROWS, :], dtype=np.float64)
        rowsum = blk.sum(1)
        del blk
    elif layout == "csr":
        ip = X["indptr"][: PROBE_ROWS + 1]
        d = X["data"][int(ip[0]): int(ip[-1])]
        rowsum = np.add.reduceat(np.asarray(d, dtype=np.float64),
                                 (ip[:-1] - ip[0]).astype(np.int64)) if len(d) else np.zeros(PROBE_ROWS)
        # reduceat mis-handles empty rows; recompute defensively
        counts = np.diff(ip).astype(np.int64)
        rowsum = np.zeros(len(counts))
        pos = 0
        dd = np.asarray(d, dtype=np.float64)
        for i, c in enumerate(counts):
            rowsum[i] = dd[pos:pos + c].sum()
            pos += c
    else:
        rowsum = np.array([])

    if rowsum.size:
        report["library_size_probe_slice"] = {
            "n_cells_probed": int(rowsum.size),
            "median": float(np.median(rowsum)), "mean": float(rowsum.mean()),
            "min": float(rowsum.min()), "max": float(rowsum.max()),
            "constant_across_cells": bool(np.ptp(rowsum) / max(np.median(rowsum), 1e-9) < 1e-3),
        }

    # ---- 4. obs ------------------------------------------------------------
    cols = obs_columns(h5)
    report["obs_columns"] = cols
    tg_key = "obs/target_gene" if "target_gene" in cols else None
    if tg_key is None:
        cand = [c for c in cols if "target" in c.lower() or "pert" in c.lower() or "guide" in c.lower()]
        report["FATAL_target_gene_missing"] = {"candidates": cand}
        tg_key = f"obs/{cand[0]}" if cand else None
    target = read_str_array(h5, tg_key) if tg_key else None
    batch = read_str_array(h5, "obs/batch") if "batch" in cols else None

    # ---- 5. var ------------------------------------------------------------
    vg = h5["var"]
    vcols = list(vg.attrs.get("column-order", []))
    vcols = [c.decode() if isinstance(c, bytes) else str(c) for c in vcols]
    idx_key = vg.attrs.get("_index", b"_index")
    idx_key = idx_key.decode() if isinstance(idx_key, bytes) else str(idx_key)
    gene_names = read_str_array(h5, f"var/{idx_key}")
    report["var"] = {
        "index_key": idx_key, "columns": vcols, "n_genes": int(len(gene_names)),
        "index_examples": gene_names[:5].tolist(),
        "gene_id_examples": read_str_array(h5, "var/gene_id")[:5].tolist() if "gene_id" in vcols else None,
    }

# ------------------------------------------------------------- summaries ----
if target is not None:
    vc = pd.Series(target).value_counts()
    ctrl_names = [c for c in vc.index if c.lower().replace("_", "-") in
                  ("non-targeting", "nontargeting", "ntc", "control")]
    ctrl = ctrl_names[0] if ctrl_names else None
    perts = vc.drop(index=ctrl) if ctrl else vc
    q = perts.quantile([0, .05, .25, .5, .75, .95, 1]).to_dict()
    report["target_gene"] = {
        "n_levels": int(vc.size),
        "control_label": ctrl,
        "n_control_cells": int(vc[ctrl]) if ctrl else None,
        "n_perturbations": int(perts.size),
        "n_perturbed_cells": int(perts.sum()),
        "cells_per_pert": {
            "mean": float(perts.mean()), "median": float(perts.median()),
            "min": int(perts.min()), "max": int(perts.max()),
            "quantiles": {str(k): float(v) for k, v in q.items()},
            "n_below_200": int((perts < 200).sum()),
            "n_below_500": int((perts < 500).sum()),
            "n_below_1000": int((perts < 1000).sum()),
            "coefficient_of_variation": float(perts.std() / perts.mean()),
        },
        "smallest_10": perts.tail(10).to_dict(),
        "largest_5": perts.head(5).to_dict(),
    }
    # per-half achievable depth = floor(cells_per_pert / 2)
    half = (perts // 2)
    report["target_gene"]["max_depth_per_half"] = {
        "median": float(half.median()),
        "n_perts_with_ge_100": int((half >= 100).sum()),
        "n_perts_with_ge_250": int((half >= 250).sum()),
        "n_perts_with_ge_500": int((half >= 500).sum()),
        "n_perts_with_ge_1000": int((half >= 1000).sum()),
    }

if target is not None and batch is not None:
    ct = pd.crosstab(pd.Series(target, name="target_gene"), pd.Series(batch, name="batch"))
    p = ct.div(ct.sum(1), axis=0).values
    with np.errstate(divide="ignore", invalid="ignore"):
        ent = -np.nansum(np.where(p > 0, p * np.log(p), 0.0), axis=1)
    ent_max = np.log(ct.shape[1])
    report["batch"] = {
        "n_batches": int(ct.shape[1]),
        "batch_labels": [str(c) for c in ct.columns[:20]],
        "batches_per_perturbation_median": float((ct > 0).sum(1).median()),
        "batches_per_perturbation_min": int((ct > 0).sum(1).min()),
        "normalised_entropy_median": float(np.median(ent / ent_max)) if ent_max > 0 else None,
        "n_perts_confined_to_single_batch": int(((ct > 0).sum(1) == 1).sum()),
        "WARNING_batch_confounded": bool(((ct > 0).sum(1) == 1).sum() > 0),
    }
    meta = pd.DataFrame({"target_gene": target, "batch": batch})
else:
    meta = pd.DataFrame({"target_gene": target if target is not None else []})

meta.to_csv(os.path.join(OUT, "v5g0_vcc_percell_meta_v1.csv.gz"), index=False,
            compression="gzip")

with open(os.path.join(OUT, "v5g0_vcc_probe_v1.json"), "w") as f:
    json.dump(report, f, indent=2, default=str)

print(json.dumps(report, indent=2, default=str))
print("\nWROTE:", os.path.join(OUT, "v5g0_vcc_probe_v1.json"))
print("WROTE:", os.path.join(OUT, "v5g0_vcc_percell_meta_v1.csv.gz"))
print("\n>>> X_call =", report.get("X_call"),
      "| layout =", report["matrix"]["layout"],
      "-> set VCC_X_MODE accordingly in STEP 1.")